In [ ]:
!pip install thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.6 MB/s eta 0:00:00


In [ ]:
!pip install phonetics

  Preparing metadata (setup.py) ... done
  Created wheel for phonetics: filename=phonetics-1.0.5-py2.py3-none-any.whl size=8696 sha256=0af10808cdaeccd3868fbbafbff8e1c686c1c06df1b0fedd3b811a1e40b7cdae
  Stored in directory: /root/.cache/pip/wheels/40/63/73/d2c3bed2dc6df7c34ec9e1402761bed3519a4c0d152858b190
Successfully built phonetics


In [ ]:
import pandas as pd
from thefuzz import fuzz
from phonetics import metaphone

# Function to preprocess names
def preprocess_name(name):
    if pd.isna(name):  # Handle missing values
        return ""
    return name.lower().strip()

# Function to check if two names are the same
def is_same_name(name1, name2):
    name1 = preprocess_name(name1)
    name2 = preprocess_name(name2)

    # Exact match
    if name1 == name2:
        return True

    # Phonetic similarity
    if metaphone(name1) == metaphone(name2):
        return True

    # Fuzzy match
    if fuzz.ratio(name1, name2) > 50:
        return True

    return False

# Function to resolve disambiguates in a list of names
def resolve_disambiguates(names):
    unique_names = []
    for name in names:
        if not any(is_same_name(name, unique_name) for unique_name in unique_names):
            unique_names.append(name)
    return unique_names

# Load the CSV file
input_csv = "/content/combined_output_standardized.csv"  # Replace with your CSV file path
df = pd.read_csv(input_csv)

# Combine all author columns into a single list of names
author_columns = ["Awardee", "Author_1", "Author_2", "Author_3", "Author_4",
                  "Author_5", "Author_6", "Author_7", "Author_8", "Author_9"]

# Resolve disambiguates for each row
for index, row in df.iterrows():
    all_names = [row[col] for col in author_columns if pd.notna(row[col])]
    unique_names = resolve_disambiguates(all_names)

    # Update the row with resolved names
    for i, col in enumerate(author_columns):
        df.at[index, col] = unique_names[i] if i < len(unique_names) else ""

# Save the cleaned data to a new CSV file
output_csv = "cleaned_file.csv"  # Replace with your desired output file path
df.to_csv(output_csv, index=False)

print(f"Cleaned data saved to {output_csv}")

<ipython-input-9-4558d5582ad4>:40: DtypeWarning: Columns (33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


Cleaned data saved to cleaned_file.csv


In [ ]:
import pandas as pd
from collections import Counter
from thefuzz import fuzz
from phonetics import metaphone

# Function to preprocess names
def preprocess_name(name):
    if pd.isna(name):
        return ""
    return name.lower().strip()

# Function to check if two names are similar
def is_same_name(name1, name2):
    name1 = preprocess_name(name1)
    name2 = preprocess_name(name2)

    if name1 == name2:
        return True  # Exact match

    if metaphone(name1) == metaphone(name2):
        return True  # Phonetic similarity

    if fuzz.ratio(name1, name2) > 50:  # Increase threshold for better accuracy
        return True  # Fuzzy match

    return False

# Function to select the best representative name
def select_best_name(names, awardee_set):
    name_counts = Counter(names)  # Count occurrences of each name
    sorted_names = sorted(names, key=lambda x: (-name_counts[x], x))  # Sort by frequency

    # Prioritize names in the awardee column
    for name in sorted_names:
        if name in awardee_set:
            return name

    return sorted_names[0]  # Otherwise, return the most frequent name

# Load the CSV file
input_csv = "/content/combined_output_standardized.csv"  # Replace with your file path
df = pd.read_csv(input_csv)

# Define author columns
author_columns = ["Awardee", "Author_1", "Author_2", "Author_3", "Author_4",
                  "Author_5", "Author_6", "Author_7", "Author_8", "Author_9"]

# Extract awardee names as a set
awardee_set = set(df["Awardee"].dropna().unique())

# Process each row
for index, row in df.iterrows():
    all_names = [row[col] for col in author_columns if pd.notna(row[col])]
    grouped_names = []

    # Group similar names
    for name in all_names:
        matched = False
        for group in grouped_names:
            if is_same_name(name, group[0]):
                group.append(name)
                matched = True
                break
        if not matched:
            grouped_names.append([name])

    # Select the best name for each group
    resolved_names = [select_best_name(group, awardee_set) for group in grouped_names]

    # Update the row with resolved names
    for i, col in enumerate(author_columns):
        df.at[index, col] = resolved_names[i] if i < len(resolved_names) else ""

# Save the cleaned data to a new CSV file
output_csv = "cleaned_file1.csv"
df.to_csv(output_csv, index=False)

print(f"Cleaned data saved to {output_csv}")


<ipython-input-11-def48bcb054f>:42: DtypeWarning: Columns (33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


Cleaned data saved to best_name.csv
